# Round 7: longer training

The R6_base recipe at 400 epochs. Only the length changes. This takes roughly 7 hours.

0. Add a Colab secret named **KAGGLE_API_TOKEN** (key icon) with Notebook access on.
1. Choose **Runtime → Change runtime type → A100** (normal RAM).
2. Choose **Runtime → Run all** and allow Google Drive access.
3. The last cell shuts the runtime down. Results, checkpoints and logs are in `MyDrive/resnet-5m/round7/`.

If it disconnects, run all again: training resumes from the last copy on Drive, at most about 10 minutes back.

In [ ]:
%cd /content
!mkdir -p resnet-5m
%cd /content/resnet-5m
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
!pip install -q -U kaggle
!mkdir -p /content/teachers
for f in ['S2_highres.pt', 'KD_C_aug.pt', 'C_ta.pt', 'C_mix.pt', 'C_ta_mix.pt', 'C_ta_mix_noerase.pt', 'A_nose_ta_mix.pt']:
    !kaggle datasets download shubhamtanwar72/resnet-5m-teachers -f {f} -p /content/teachers -q
!kaggle datasets download shubhamtanwar72/resnet-5m-s4 -f S4_deep.pt -p /content/teachers -q
for f in ['KD5_full.pt', 'KD5_full_cutout.pt']:
    !kaggle datasets download shubhamtanwar72/resnet-5m-kd5full -f {f} -p /content/teachers -q
!mkdir -p data
!kaggle datasets download shubhamtanwar72/resnet-5m-cifar10 -f cifar10.bin -p data -q
!mv data/cifar10.bin data/cifar-10-python.tar.gz
!python -c "from torchvision import datasets; datasets.CIFAR10('data', download=True); print('CIFAR-10 ready')"
!ls -la /content/teachers

In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.ops import stochastic_depth

MAX_PARAMS = 5_000_000
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)


class SqueezeExcite(nn.Module):
    """Channel attention (Hu et al., 2018): reweight channels using globally pooled context."""

    def __init__(self, c, reduction=16):
        super().__init__()
        self.fc1 = nn.Conv2d(c, c // reduction, 1)
        self.fc2 = nn.Conv2d(c // reduction, c, 1)

    def forward(self, x):
        s = F.adaptive_avg_pool2d(x, 1)
        return x * torch.sigmoid(self.fc2(F.relu(self.fc1(s))))


class PreActBlock(nn.Module):
    """Pre-activation basic block (He et al., 2016): BN-ReLU-conv x2, keeping the skip path a clean identity.

    Downsampling shortcuts use avg-pool then 1x1 conv (ResNet-D, He et al., 2019) so no activations are discarded.
    """

    def __init__(self, cin, cout, stride=1, se=True, drop_path=0.0):
        super().__init__()
        self.drop_path = drop_path
        self.bn1 = nn.BatchNorm2d(cin)
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.se = SqueezeExcite(cout) if se else nn.Identity()
        self.shortcut = None
        if stride != 1 or cin != cout:
            pool = [nn.AvgPool2d(stride)] if stride != 1 else []
            self.shortcut = nn.Sequential(*pool, nn.Conv2d(cin, cout, 1, bias=False))

    def forward(self, x):
        out = F.relu(self.bn1(x))
        skip = x if self.shortcut is None else self.shortcut(out)
        out = self.conv2(F.relu(self.bn2(self.conv1(out))))
        return stochastic_depth(self.se(out), self.drop_path, "row", self.training) + skip


class PreActBottleneck(nn.Module):
    """Pre-activation bottleneck (He et al., 2016): 1x1 down, 3x3, 1x1 up, for more depth per parameter."""

    def __init__(self, cin, cout, stride=1, se=True, reduction=4, drop_path=0.0):
        super().__init__()
        self.drop_path = drop_path
        mid = cout // reduction
        self.bn1 = nn.BatchNorm2d(cin)
        self.conv1 = nn.Conv2d(cin, mid, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(mid)
        self.conv2 = nn.Conv2d(mid, mid, 3, stride, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(mid)
        self.conv3 = nn.Conv2d(mid, cout, 1, bias=False)
        self.se = SqueezeExcite(cout) if se else nn.Identity()
        self.shortcut = None
        if stride != 1 or cin != cout:
            pool = [nn.AvgPool2d(stride)] if stride != 1 else []
            self.shortcut = nn.Sequential(*pool, nn.Conv2d(cin, cout, 1, bias=False))

    def forward(self, x):
        out = F.relu(self.bn1(x))
        skip = x if self.shortcut is None else self.shortcut(out)
        out = self.conv2(F.relu(self.bn2(self.conv1(out))))
        out = self.conv3(F.relu(self.bn3(out)))
        return stochastic_depth(self.se(out), self.drop_path, "row", self.training) + skip


BLOCKS = {"basic": PreActBlock, "bottleneck": PreActBottleneck}


class ResNet(nn.Module):
    """CIFAR ResNet: 3x3 stem, then one stage per width; each stage after the first halves resolution.

    Takes images in [0, 1]; normalization is done inside the model so inference needs no extra preprocessing.
    """

    def __init__(self, widths=(64, 128, 256), blocks=(3, 4, 3), se=True, num_classes=10, block="basic", drop_path=0.0):
        super().__init__()
        self.register_buffer("mean", torch.tensor(CIFAR_MEAN).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(CIFAR_STD).view(1, 3, 1, 1))
        self.stem = nn.Conv2d(3, widths[0], 3, 1, 1, bias=False)
        layers, cin, total = [], widths[0], sum(blocks)
        for i, (w, n) in enumerate(zip(widths, blocks)):
            for j in range(n):
                # Stochastic depth grows linearly with depth (Huang et al., 2016).
                rate = drop_path * len(layers) / max(total - 1, 1)
                layers.append(BLOCKS[block](cin, w, stride=2 if i > 0 and j == 0 else 1, se=se, drop_path=rate))
                cin = w
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(cin)
        self.fc = nn.Linear(cin, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")

    def forward(self, x):
        x = self.layers(self.stem((x - self.mean) / self.std))
        x = F.relu(self.bn(x))
        return self.fc(torch.flatten(F.adaptive_avg_pool2d(x, 1), 1))


def tta_logits(model, x, shift=0):
    """Average logits over the horizontal flip and, if shift > 0, translations of `shift` pixels in each direction."""
    size = x.size(-1)
    padded = F.pad(x, (shift,) * 4, "reflect")
    offsets = [(0, 0)] + ([(shift, 0), (-shift, 0), (0, shift), (0, -shift)] if shift else [])
    out = 0
    for dx, dy in offsets:
        view = padded[..., shift + dy:shift + dy + size, shift + dx:shift + dx + size]
        out = out + model(view) + model(view.flip(3))
    return out / (2 * len(offsets))


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
%%writefile train.py
import argparse
import math
import os
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms as T
from torchvision.transforms import v2

from model import BLOCKS, MAX_PARAMS, ResNet, count_params


def get_args():
    p = argparse.ArgumentParser(description="Train a <5M-param ResNet on CIFAR-10")
    p.add_argument("--widths", type=int, nargs="+", default=[64, 128, 256])
    p.add_argument("--blocks", type=int, nargs="+", default=[3, 4, 3])
    p.add_argument("--block", choices=BLOCKS, default="basic", help="residual block type")
    p.add_argument("--se", action=argparse.BooleanOptionalAction, default=True, help="squeeze-and-excitation")
    p.add_argument("--epochs", type=int, default=200)
    p.add_argument("--batch-size", type=int, default=256)
    p.add_argument("--lr", type=float, default=0.2)
    p.add_argument("--wd", type=float, default=5e-4)
    p.add_argument("--warmup-epochs", type=int, default=5)
    p.add_argument("--label-smoothing", type=float, default=0.1)
    p.add_argument("--autoaugment", action="store_true", help="add TrivialAugmentWide")
    p.add_argument("--mix", type=float, default=0.0, help="probability of applying CutMix or MixUp to a batch")
    p.add_argument("--erase-p", type=float, default=0.5, help="RandomErasing (cutout) probability")
    p.add_argument("--cutout", action="store_true", help="erase a 16x16 square from every image (overrides --erase-p)")
    p.add_argument("--drop-path", type=float, default=0.0, help="stochastic depth rate of the last block")
    p.add_argument("--ema", type=float, default=0.0, help="weight EMA decay per step; evaluates and saves the EMA model")
    p.add_argument("--sam", type=float, default=0.0, help="SAM neighborhood size rho (about 2x the compute)")
    p.add_argument("--teacher", nargs="+", default=[], help="checkpoint(s) to distill from (logits are averaged)")
    p.add_argument("--teacher-flip", action="store_true", help="also average teacher logits over the horizontal flip")
    p.add_argument("--kd-alpha", type=float, default=0.9, help="weight of the distillation loss vs. the label loss")
    p.add_argument("--kd-temp", type=float, default=4.0)
    p.add_argument("--compile", action="store_true")
    p.add_argument("--workers", type=int, default=8)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--data-dir", default="data")
    p.add_argument("--out", default="checkpoints/model.pt")
    p.add_argument("--resume", action="store_true", help="continue from --out if it exists (e.g. after preemption)")
    return p.parse_args()


def loaders(args):
    aug = [T.RandomCrop(32, padding=4, padding_mode="reflect"), T.RandomHorizontalFlip()]
    if args.autoaugment:
        aug.append(T.TrivialAugmentWide())
    erase = dict(p=1.0, scale=(0.25, 0.25), ratio=(1, 1)) if args.cutout else dict(p=args.erase_p, scale=(0.02, 0.2))
    aug += [T.ToTensor(), T.RandomErasing(value=0.5, **erase)]
    train = datasets.CIFAR10(args.data_dir, train=True, download=True, transform=T.Compose(aug))
    # The official CIFAR-10 test split is used only to monitor generalization, never for training.
    test = datasets.CIFAR10(args.data_dir, train=False, download=True, transform=T.ToTensor())
    kw = dict(num_workers=args.workers, pin_memory=True, persistent_workers=args.workers > 0)
    return (
        DataLoader(train, args.batch_size, shuffle=True, drop_last=True, **kw),
        DataLoader(test, 1024, **kw),
    )


@torch.no_grad()
def evaluate(model, loader, device, amp, amp_dtype):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device.type, amp_dtype, enabled=amp):
            correct += (model(x).argmax(1) == y).sum().item()
        total += y.numel()
    return correct / total


def main():
    args = get_args()
    torch.manual_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    amp = device.type == "cuda"
    # Older GPUs such as the V100 have no bf16; float16 needs loss scaling to avoid gradient underflow.
    amp_dtype = torch.bfloat16 if amp and torch.cuda.is_bf16_supported() else torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=amp and amp_dtype == torch.float16)
    torch.backends.cudnn.benchmark = True

    assert not (args.sam and scaler.is_enabled()), "--sam is not supported with float16 loss scaling"

    cfg = dict(widths=args.widths, blocks=args.blocks, se=args.se, block=args.block, drop_path=args.drop_path)
    model = ResNet(**cfg).to(device, memory_format=torch.channels_last)
    n_params = count_params(model)
    print(f"params: {n_params:,}")
    assert n_params <= MAX_PARAMS, f"model has {n_params:,} params, limit is {MAX_PARAMS:,}"
    net = torch.compile(model) if args.compile else model
    ema = None
    if args.ema:
        ema = torch.optim.swa_utils.AveragedModel(
            model, multi_avg_fn=torch.optim.swa_utils.get_ema_multi_avg_fn(args.ema), use_buffers=True
        )
    bns = [m for m in model.modules() if isinstance(m, nn.BatchNorm2d)]

    train_loader, test_loader = loaders(args)
    # No weight decay on BN and bias parameters.
    decay = [p for p in model.parameters() if p.ndim > 1]
    no_decay = [p for p in model.parameters() if p.ndim <= 1]
    opt = torch.optim.SGD(
        [{"params": decay, "weight_decay": args.wd}, {"params": no_decay, "weight_decay": 0}],
        lr=args.lr, momentum=0.9, nesterov=True,
    )
    steps, warmup = args.epochs * len(train_loader), args.warmup_epochs * len(train_loader)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: s / warmup if s < warmup else 0.5 * (1 + math.cos(math.pi * (s - warmup) / (steps - warmup)))
    )
    criterion = nn.CrossEntropyLoss(label_smoothing=args.label_smoothing)
    mix = v2.RandomChoice([v2.CutMix(num_classes=10), v2.MixUp(num_classes=10)])
    teachers = []
    for path in args.teacher:
        t = torch.load(path, map_location=device)
        teachers.append(ResNet(**t["cfg"]).to(device, memory_format=torch.channels_last).eval())
        teachers[-1].load_state_dict(t["model"])

    out = Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    start = 0
    if args.resume and out.exists():
        ckpt = torch.load(out, map_location=device)
        model.load_state_dict(ckpt.get("raw", ckpt["model"]))
        if ema:
            ema.load_state_dict(ckpt["ema"])
        opt.load_state_dict(ckpt["opt"])
        sched.load_state_dict(ckpt["sched"])
        if "scaler" in ckpt:
            scaler.load_state_dict(ckpt["scaler"])
        start = ckpt["epoch"]
        print(f"resumed from {out} at epoch {start}")
    for epoch in range(start, args.epochs):
        net.train()
        t0, loss_sum = time.time(), torch.zeros((), device=device)
        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            if torch.rand(()) < args.mix:
                x, y = mix(x, y)  # y becomes soft labels
            x = x.contiguous(memory_format=torch.channels_last)
            target = None
            if teachers:
                # The teacher labels the exact same augmented batch the student sees.
                with torch.no_grad(), torch.autocast(device.type, amp_dtype, enabled=amp):
                    target = torch.stack([t(x) for t in teachers]).float().mean(0)
                    if args.teacher_flip:
                        target = (target + torch.stack([t(x.flip(3)) for t in teachers]).float().mean(0)) / 2

            def compute_loss():
                with torch.autocast(device.type, amp_dtype, enabled=amp):
                    logits = net(x)
                    loss = criterion(logits, y)
                    if target is not None:
                        T = args.kd_temp
                        kd = F.kl_div(F.log_softmax(logits.float() / T, 1), F.softmax(target / T, 1), reduction="batchmean")
                        loss = args.kd_alpha * kd * T * T + (1 - args.kd_alpha) * loss
                return loss

            loss = compute_loss()
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if args.sam:
                # SAM (Foret et al., 2021): take the gradient at the worst nearby weights, then step from the original ones.
                params = [p for p in model.parameters() if p.grad is not None]
                scale = args.sam / (torch.stack([p.grad.norm() for p in params]).norm() + 1e-12)
                perturb = [p.grad * scale for p in params]
                with torch.no_grad():
                    torch._foreach_add_(params, perturb)
                for m in bns:
                    m.momentum = 0  # the batch was already counted in the BN statistics
                opt.zero_grad(set_to_none=True)
                compute_loss().backward()
                for m in bns:
                    m.momentum = 0.1
                with torch.no_grad():
                    torch._foreach_sub_(params, perturb)
            scaler.step(opt)
            scaler.update()
            sched.step()
            if ema:
                ema.update_parameters(model)
            loss_sum += loss.detach()
        acc = evaluate(ema if ema else net, test_loader, device, amp, amp_dtype)
        print(f"epoch {epoch + 1}/{args.epochs}  loss {loss_sum.item() / len(train_loader):.4f}  "
              f"test_acc {acc:.4f}  lr {sched.get_last_lr()[0]:.4f}  {time.time() - t0:.1f}s", flush=True)
        state = {"model": (ema.module if ema else model).state_dict(), "cfg": cfg, "opt": opt.state_dict(), "sched": sched.state_dict(),
                 "scaler": scaler.state_dict(), "epoch": epoch + 1, "test_acc": acc}
        if ema:
            state |= {"raw": model.state_dict(), "ema": ema.state_dict()}
        torch.save(state, out.with_suffix(".tmp"))
        os.replace(out.with_suffix(".tmp"), out)  # atomic, so preemption mid-save can't corrupt the checkpoint
    print(f"saved {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile evaluate.py
import argparse

import torch
from torchvision import datasets, transforms as T

from model import ResNet, tta_logits


@torch.no_grad()
def main():
    p = argparse.ArgumentParser(description="CIFAR-10 test accuracy (with flip TTA) of checkpoints and of their logit-averaged ensemble")
    p.add_argument("checkpoints", nargs="+")
    p.add_argument("--data-dir", default="data")
    p.add_argument("--shift", type=int, default=0, help="also average over translations of this many pixels")
    args = p.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    test = datasets.CIFAR10(args.data_dir, train=False, download=True, transform=T.ToTensor())
    x = torch.stack([img for img, _ in test]).to(device)
    y = torch.tensor(test.targets, device=device)

    logits = []
    for path in args.checkpoints:
        ckpt = torch.load(path, map_location=device)
        model = ResNet(**ckpt["cfg"]).to(device).eval()
        model.load_state_dict(ckpt["model"])
        out = torch.cat([tta_logits(model, b, args.shift) for b in x.split(500)])
        logits.append(out)
        print(f"{path}: {(out.argmax(1) == y).float().mean():.4f}")
    if len(logits) > 1:
        print(f"ensemble of {len(logits)}: {(torch.stack(logits).mean(0).argmax(1) == y).float().mean():.4f}")


if __name__ == "__main__":
    main()

In [ ]:
%%bash
cd /content/resnet-5m
OUT=/content/round7
DRIVE=/content/drive/MyDrive/resnet-5m/round7
mkdir -p $OUT $DRIVE
cp -n $DRIVE/* $OUT/ 2>/dev/null   # a new runtime resumes from the last copy on Drive
A="--widths 128 192 232 --blocks 4 3 2 --no-se --autoaugment --mix 0.5 --erase-p 0 --cutout --epochs 400 --workers 10 --seed 1 --teacher-flip --resume"
T=/content/teachers
TF="--teacher $T/S2_highres.pt $T/KD_C_aug.pt $T/C_ta.pt $T/C_mix.pt $T/C_ta_mix.pt $T/C_ta_mix_noerase.pt $T/A_nose_ta_mix.pt $T/S4_deep.pt $T/KD5_full.pt $T/KD5_full_cutout.pt"

save() {
  for f in $OUT/*.pt $OUT/*.log; do
    [ -f "$f" ] && cp "$f" "$DRIVE/.tmp" && mv "$DRIVE/.tmp" "$DRIVE/$(basename $f)"
  done
}

# These runs last hours, so progress is copied to Drive every 10 minutes.
(while sleep 600; do save; done) &
SAVER=$!

run() {
  n=$1; shift
  python -u train.py "$@" $A --out $OUT/$n.pt 2>&1 | tee -a $OUT/$n.log \
    | grep --line-buffered -E 'epoch [0-9]*0/|Traceback|Error' | sed -u "s/^/$n  /"
  echo "$n finished"
}

run R7_long $TF
kill $SAVER
save
echo "all runs copied to Drive"

In [ ]:
import glob, os, re, subprocess
lines = [f"{'run':14}{'epochs':>7}{'final':>8}{'best':>8}{'s/epoch':>9}"]
for f in sorted(glob.glob('/content/round7/*.log')):
    r = re.findall(r'epoch (\d+)/\d+  loss \S+  test_acc (\S+)  lr \S+  ([\d.]+)s', open(f).read())
    if r:
        acc = [100 * float(x[1]) for x in r]
        t = sorted(float(x[2]) for x in r)[len(r) // 2]
        lines.append(f"{os.path.basename(f)[:-4]:14}{r[-1][0]:>7}{acc[-1]:8.2f}{max(acc):8.2f}{t:9.1f}")
pts = sorted(glob.glob('/content/round7/*.pt'))
lines.append(subprocess.run(['python', 'evaluate.py', *pts], capture_output=True, text=True).stdout)
report = '\n'.join(lines)
print(report)
open('/content/drive/MyDrive/resnet-5m/round7/results.txt', 'w').write(report)

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
runtime.unassign()  # stops the paid runtime; everything is already on Drive